## Set up AnnData object with integration winner, and plot UMAP

In [1]:
# Path and system utilities
import os                    # Operating system interface
import sys                   # System-specific parameters and functions
import glob                  # File pattern matching
from pathlib import Path     # Object-oriented filesystem paths
from pyhere import here      # Reproducible project paths

# Single-cell data handling
import anndata               # Core data structure for single-cell data
import scanpy as sc          # Analysis and visualization of single-cell data


# parallel processing
from joblib import parallel_backend  # Parallel computing support


# Miscellaneous utilities
import warnings              # Suppress or manage warnings

# Custom modules and functions
sys.path.append(str(here('scripts/misc')))  # Add custom script path to system
import my_anndata as ma                    # Custom AnnData utilities

Parameters

In [2]:
base_dir = str(here('data/integrate/second_pass/'))
plot_dir = os.path.join(base_dir, 'plot') 
objects_dir = os.path.join(base_dir, 'objects') 
anndata_dir = str(here('data/anndata/'))

Import data

In [3]:
adata_full = ad.read_h5ad(os.path.join(anndata_dir, "AC_combined.h5ad"))

Remove information we dont need anymore from the Anndata object

In [4]:
to_delete_obsm = [x for x in adata_full.obsm.keys() if not x == "X_latent_2"]
to_delete_uns = [
    x
    for x in adata_full.uns.keys()
    if x.endswith("umap") or x.endswith("neighbors") or x.endswith("colors")
]
to_delete_obsp = [
    x
    for x in adata_full.obsp.keys()
    if x.endswith("distances") or x.endswith("connectivities")
]
to_delete_obs = ["technical", "technical_donor"]

for key in to_delete_obsm:
    del adata_full.obsm[key]

for key in to_delete_uns:
    del adata_full.uns[key]

for key in to_delete_obsp:
    del adata_full.obsp[key]

for key in to_delete_obs:
    del adata_full.obs[key]

Rename columns so that names are more meaningfull

In [5]:
# Save old meta data
old_meta = adata_full.obs.copy()

# Rename
adata_full.obs.rename(columns={'ic_id_study': 'ic_id_dataset'}, inplace=True) # Current id reflects dataset not study
adata_full.obs.rename(columns={'ic_id_study_overall': 'ic_id_study'}, inplace=True) # Now overall study can be called study
adata_full.obs.rename(columns={'ic_id_donor': 'ic_id_dataset_donor'}, inplace=True) # Now id reflects donor per dataset

# Check renaming was done correctly
assert (old_meta['ic_id_study'] == adata_full.obs['ic_id_dataset']).all()
assert (old_meta['ic_id_study_overall'] == adata_full.obs['ic_id_study']).all()
assert (old_meta['ic_id_donor'] == adata_full.obs['ic_id_dataset_donor']).all()

Find neighbors for X_latent_2

In [6]:
with parallel_backend("threading", n_jobs=100):
    sc.pp.neighbors(adata_full, use_rep='X_latent_2', key_added='X_latent_2_neighbors')

Generate UMAP

In [7]:
sc.tl.umap(
    adata_full, 
    neighbors_key='X_latent_2_neighbors', 
    key_added='X_latent_2_umap'
)

Save AnnData object

In [8]:
adata_full.write(os.path.join(anndata_dir, 'AD_combined.h5ad'))